In [56]:
def visualize_str(nanoribbon):
    import nglview as nv
    import numpy as np

    w = nv.show_ase(nanoribbon)
    w.add_ball_and_stick()
    w.add_label(color="black", labelType="atomindex")

    origin = nanoribbon.positions.min(axis=0) - np.array([2.0, 2.0, 2.0])
    L = 2.0

    # стрелки
    w.shape.add_arrow(origin, origin + [L, 0, 0], [1, 0, 0], 0.2)  # X (red)
    w.shape.add_arrow(origin, origin + [0, L, 0], [0, 1, 0], 0.2)  # Y (green)
    w.shape.add_arrow(origin, origin + [0, 0, L], [0, 0, 1], 0.2)  # Z (blue)

    # подписи
    w.shape.add_text(origin + [L, 0, 0], [1, 0, 0], 1.0, "Z")
    w.shape.add_text(origin + [0, L, 0], [0, 1, 0], 1.0, "X")
    w.shape.add_text(origin + [0, 0, L], [0, 0, 1], 1.0, "Y")

    return w

In [57]:
def chiral_nanoribbon(n, m, length=1):
    from ase.build import nanotube
    import numpy as np

    cnt = nanotube(n=n, m=m, length=length)

    bond_lenght = 1.42
    a = np.sqrt(3) * bond_lenght
    R = a * np.sqrt(n ** 2 + n * m + m ** 2) / (2 * np.pi)

    for i in range(len(cnt.positions)):
        x, y, z = cnt.positions[i]

        phi0 = 0.1 #сдвиг для более аккуратного разреза
        phi = np.arctan2(y, x)                # (-pi, pi]
        phi = (phi - phi0 + 2 * np.pi) % (2 * np.pi) # -> [0, 2pi) 
        # - необходимо, чтобы при переводе координат крайние атомы оказывались в противополложных концах ленты

        x_new = R * phi
        y_new = 0
        z_new = z

        cnt.positions[i] = np.array([x_new, y_new, z_new])
    
    cnt.set_cell([0.0, 0.0, cnt.cell.lengths()[2]])
    cnt.pbc = (False, False, True)
    
    return cnt

In [58]:
def neighbor_list(nanoribbon, cutoff=1.85):
    from ase.neighborlist import neighbor_list

    i, j, S = neighbor_list('ijS', nanoribbon, cutoff)
    cell = nanoribbon.cell.array
    pos = nanoribbon.positions

    neigh = {a: [] for a in range(len(nanoribbon))}
    for ia, ja, Sa in zip(i, j, S):
        rj_img = pos[ja] + Sa @ cell  # координаты соседа с учётом трансляции
        neigh[int(ia)].append((int(ja), tuple(int(x) for x in Sa), rj_img))

    return neigh

In [59]:
chiral_rib = chiral_nanoribbon(n=10, m=5, length=1)

In [60]:
visualize_str(chiral_rib)

NGLWidget()

In [ ]:
# трансляция вдоль оси y
neigh = neighbor_list(chiral_rib)
pos = chiral_rib.positions

for ia in range(len(chiral_rib)):
    print(f"Atom {ia:3d}  r=({pos[ia][0]:8.4f},{pos[ia][1]:8.4f},{pos[ia][2]:8.4f})  neighbors={len(neigh[ia])}")
    for ja, Sa, rj in neigh[ia]:
        print(f"   -> j={ja:3d}, S={Sa}, rj=({rj[0]:8.4f},{rj[1]:8.4f},{rj[2]:8.4f})")

Atom   0  r=( 32.0185,  0.0000,  0.0000)  neighbors=2
   -> j=133, S=(0, 0, 0), rj=( 31.5537,  0.0000,  1.3418)
   -> j=131, S=(0, 0, -1), rj=( 31.0888,  0.0000, -1.0734)
Atom   1  r=(  0.8766,  0.0000, 11.0025)  neighbors=2
   -> j=  8, S=(0, 0, 0), rj=(  1.3414,  0.0000,  9.6608)
   -> j= 10, S=(0, 0, 1), rj=(  1.8062,  0.0000, 12.0760)
Atom   2  r=( 32.4833,  0.0000,  2.4152)  neighbors=2
   -> j=135, S=(0, 0, 0), rj=( 32.0185,  0.0000,  3.7570)
   -> j=133, S=(0, 0, 0), rj=( 31.5537,  0.0000,  1.3418)
Atom   3  r=(  1.3414,  0.0000,  2.1468)  neighbors=2
   -> j= 12, S=(0, 0, 0), rj=(  2.2710,  0.0000,  3.2203)
   -> j= 10, S=(0, 0, 0), rj=(  1.8062,  0.0000,  0.8051)
Atom   4  r=(  0.4118,  0.0000,  4.8304)  neighbors=1
   -> j=  5, S=(0, 0, 0), rj=(  1.8062,  0.0000,  4.5620)
Atom   5  r=(  1.8062,  0.0000,  4.5620)  neighbors=3
   -> j=  4, S=(0, 0, 0), rj=(  0.4118,  0.0000,  4.8304)
   -> j= 14, S=(0, 0, 0), rj=(  2.7358,  0.0000,  5.6355)
   -> j= 12, S=(0, 0, 0), rj=(  2.271

In [ ]:
# для визуализации - резали эту трубку

from ase.build import nanotube
cnt = nanotube(n=10, m=5, length=1)

In [46]:
visualize_str(cnt)

NGLWidget()